# Kaggle submission notebook

## 1. Notebook setup
### 1.1. Imports

In [1]:
import json

import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import KNNImputer
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

### 1.2. Run configuration

In [2]:
IMPUTE = True

### 1.2. Data loading

In [3]:
train_df = pd.read_csv('../data/train.csv')
train_df.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [4]:
test_df = pd.read_csv('../data/test.csv')
test_df.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [5]:
with open("../data/schema.json", "r") as f:
    metadata = json.load(f)

features = metadata['features']
categorical_features = metadata['categorical_features']
continuous_features = metadata['continuous_features']
label = metadata['label']

## 2. Data preprocessing

### 2.1. Feature encoding

In [6]:
feature_encoder = OneHotEncoder(sparse_output=False)
feature_encoder.fit(train_df[categorical_features])

encoded_train_features = feature_encoder.transform(train_df[categorical_features])
encoded_test_features = feature_encoder.transform(test_df[categorical_features])

encoded_train_df = pd.DataFrame(encoded_train_features, columns=feature_encoder.get_feature_names_out())
encoded_test_df = pd.DataFrame(encoded_test_features, columns=feature_encoder.get_feature_names_out())

train_df = pd.concat([train_df.drop(columns=categorical_features), encoded_train_df], axis=1)
test_df = pd.concat([test_df.drop(columns=categorical_features), encoded_test_df], axis=1)

### 2.2. Label encoding

In [7]:
label_encoder = LabelEncoder()
train_df[label] = label_encoder.fit_transform(train_df[label])

### 2.3. Imputation

In [8]:
if IMPUTE:
    imputer = KNNImputer(n_neighbors=3, weights='distance')

    train_df[continuous_features] = imputer.fit_transform(train_df[continuous_features])
    test_df[continuous_features] = imputer.transform(test_df[continuous_features])

    train_df.to_csv("../data/train_imputed.csv", index=False)
    test_df.to_csv("../data/test_imputed.csv", index=False)

else:
    train_df = pd.read_csv("../data/train_imputed.csv")
    test_df = pd.read_csv("../data/test_imputed.csv")

**Note:** imputation takes about 15 minutes.

## 3. Gradient boosting classifier

In [9]:
train_df.head().T

,0,1,2,3,4
id,0.00,1.00,2.00,3.00,4.00
health_condition,2.00,0.00,2.00,2.00,0.00
sleep_duration,5.22,5.53,5.29,4.70,7.23
heart_rate,70.60,71.30,75.40,77.20,73.40
bmi,25.66,25.84,24.54,23.13,28.44
calorie_expenditure,2174.00,1966.00,2688.00,2630.00,2560.00
step_count,1326.00,9891.00,14216.00,7174.00,6584.00
exercise_duration,19.80,49.90,38.10,59.90,46.00
water_intake,1.86,1.26,1.60,2.02,2.25
diet_type_balanced,0.00,0.00,0.00,0.00,0.00


In [10]:
gbc = GradientBoostingClassifier()

# Use cross validation to evaluate the model
scores = cross_val_score(
    gbc,
    train_df.drop(columns=['id', 'health_condition']),
    train_df['health_condition'], 
    cv=10,
    scoring='r2',
)

upper_bound = scores.mean() + 1.96 * scores.std() / (len(scores) ** 0.5)
lower_bound = scores.mean() - 1.96 * scores.std() / (len(scores) ** 0.5)
print(f"Cross-validation R2 scores: {scores}")
print(f"Mean cross-validation R2 score: {scores.mean()}")
print(f"95% confidence interval for the R2 score: ({lower_bound}, {upper_bound})")

Cross-validation R2 scores: [0.71024918 0.71398133 0.70520229 0.70804382 0.70732676 0.71276277
 0.71153272 0.70970885 0.7111012  0.70579914]
Mean cross-validation R2 score: 0.7095708063175198
95% confidence interval for the R2 score: (0.7078510402481857, 0.7112905723868539)


In [11]:
# Fit the model on the entire training set
gbc.fit(train_df.drop(columns=['id', 'health_condition']), train_df['health_condition'])

# Make predictions on the test set
predictions = gbc.predict(test_df.drop(columns=['id']))
print("Predictions on the test set:", predictions)

# Save predictions to a CSV file
submission_df = pd.DataFrame({'id': test_df['id'], 'health_condition': predictions})
submission_df.to_csv('submission.csv', index=False)

Predictions on the test set: [2 0 0 ... 0 0 2]


In [12]:
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'health_condition': ['at-risk'] * len(test_df)
})

submission_df.head()

,id,health_condition
0,690088,at-risk
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,at-risk


In [13]:
submission_df.to_csv('../data/submission.csv', index=False)